In [ ]:
[]

# Preprocessing and Baseline Modeling

This notebook builds the realistic preprocessing stack and the baseline bank-marketing model while respecting the business constraint that `duration` is excluded from the production feature set.

The workflow is intentionally structured to mirror a real ML pipeline:

1. import the raw Kaggle data
2. clean obvious duplicates and missing-value placeholders
3. split without leakage
4. create a `ColumnTransformer` pipeline for numeric and categorical features
5. train a logistic regression baseline
6. evaluate using business-aware metrics
7. compare against a leakage benchmark to teach the risk of post-call data


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from src.data_loader import load_bank_data
from src.preprocessing import clean_dataset

raw_df = load_bank_data('data/bank_marketing.csv')
clean_df = clean_dataset(raw_df)
print('Cleaned shape:', clean_df.shape)
print('Duplicate rows:', raw_df.duplicated().sum())
print('Missing values:', clean_df.isna().sum().sum())

X = clean_df.drop(columns=['y'])
y = clean_df['y'].str.lower().map({'yes': 1, 'no': 0})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

numeric_cols = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col])]
categorical_cols = [col for col in X.columns if col not in numeric_cols]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_cols,
        ),
    ],
    remainder='drop',
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced')),
])

model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

print('Accuracy:', round(accuracy_score(y_test, preds), 4))
print('Precision:', round(precision_score(y_test, preds, zero_division=0), 4))
print('Recall:', round(recall_score(y_test, preds, zero_division=0), 4))
print('F1:', round(f1_score(y_test, preds, zero_division=0), 4))
print('ROC-AUC:', round(roc_auc_score(y_test, probs), 4))

# Leakage benchmark: intentionally include the forbidden duration feature
leak_df = raw_df.copy()
X_leak = leak_df.drop(columns=['y'])
y_leak = leak_df['y'].str.lower().map({'yes': 1, 'no': 0})
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leak,
    y_leak,
    test_size=0.2,
    random_state=42,
    stratify=y_leak,
)

numeric_leak = [col for col in X_leak.columns if pd.api.types.is_numeric_dtype(X_leak[col])]
categorical_leak = [col for col in X_leak.columns if col not in numeric_leak]

preprocessor_leak = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_leak,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_leak,
        ),
    ],
    remainder='drop',
)

leak_model = Pipeline([
    ('preprocessor', preprocessor_leak),
    ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced')),
])

leak_model.fit(Xl_train, yl_train)
leak_probs = leak_model.predict_proba(Xl_test)[:, 1]
leak_preds = (leak_probs >= 0.5).astype(int)

print('\nLeakage benchmark metrics (including duration):')
print('Accuracy:', round(accuracy_score(yl_test, leak_preds), 4))
print('Precision:', round(precision_score(yl_test, leak_preds, zero_division=0), 4))
print('Recall:', round(recall_score(yl_test, leak_preds, zero_division=0), 4))
print('F1:', round(f1_score(yl_test, leak_preds, zero_division=0), 4))
print('ROC-AUC:', round(roc_auc_score(yl_test, leak_probs), 4))
